In [1]:
# Imports
from pathlib import Path
from experiment.utils import TrainedModel, TrainedModelID

import pandas as pd
import torch
from neuralhydrology.nh_run import start_run, eval_run, finetune
from experiment.eval import evaluate_models
import os
import yaml

In [2]:
model = TrainedModel(TrainedModelID.SOTA_20)

df = pd.read_csv(model.metrics_file, dtype={'basin':str})

basin_data = df.loc[df['NSE'] < df['NSE'].median()].sample(n=1)

basin = basin_data.basin.iloc[0]

basin='08086290'

In [6]:
# Add the path to the pre-trained model to the finetune config
file_path = "finetune.yml"

with open(file_path, "a") as fp:
    fp.write(f"\nbase_run_dir: {model.run_dir.absolute()}")

# Load the existing YAML data
with open(file_path, 'r') as f:
    data = yaml.safe_load(f)

data['experiment_name'] = f'basin_{basin}'  # Example modification

# Write back to the YAML file
with open(file_path, 'w') as f:
    yaml.dump(data, f)   


In [7]:
# Create a basin file with the basin we selected above
with open("finetune_basin.txt", "w") as fp:
    fp.write(basin)

In [8]:
# finetune
finetune(Path("finetune.yml"))

2024-09-18 18:36:54,359: Logging to /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_08086290/output.log initialized.
2024-09-18 18:36:54,360: ### Folder structure created at /home/admin/Fine-Flood-Forecasts/experiment/finetuning_playground/runs/basin_08086290
2024-09-18 18:36:54,360: ### Start finetuning with pretrained model stored in /home/admin/Fine-Flood-Forecasts/experiment/models/runs/sota_20
2024-09-18 18:36:54,361: ### Run configurations for basin_08086290
2024-09-18 18:36:54,361: additional_feature_files: None
2024-09-18 18:36:54,361: batch_size: 256
2024-09-18 18:36:54,362: checkpoint_path: None
2024-09-18 18:36:54,363: clip_gradient_norm: 1
2024-09-18 18:36:54,363: clip_targets_to_zero: ['QObs(mm/d)']
2024-09-18 18:36:54,363: commit_hash: 6dde7b4
2024-09-18 18:36:54,364: data_dir: /home/admin/Fine-Flood-Forecasts/data/CAMELS_US
2024-09-18 18:36:54,364: dataset: camels_us
2024-09-18 18:36:54,365: device: cuda:0
2024-09-18 18:36:54,365: dynamic_inp

/home/admin/Fine-Flood-Forecasts/neuralhydrology/training/basetrainer.py:160: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(torch.load(str(checkpo

# Epoch 1: 100%|██████████| 13/13 [00:00<00:00, 16.79it/s, Loss: 0.0126]
2024-09-18 18:36:57,499: Epoch 1 average loss: avg_loss: 0.01040, avg_total_loss: 0.01040
2024-09-18 18:36:57,503: Setting learning rate to 5e-05
# Epoch 2: 100%|██████████| 13/13 [00:00<00:00, 26.01it/s, Loss: 0.0014]
2024-09-18 18:36:58,006: Epoch 2 average loss: avg_loss: 0.00989, avg_total_loss: 0.00989
# Epoch 3: 100%|██████████| 13/13 [00:00<00:00, 24.89it/s, Loss: 0.0294]
2024-09-18 18:36:58,535: Epoch 3 average loss: avg_loss: 0.01033, avg_total_loss: 0.01033
# Validation: 100%|██████████| 1/1 [00:00<00:00,  1.49it/s]

/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:344: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset_one = to_offset(freq_one)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:387: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  factor = pd.to_timedelta(freq_one) / pd.to_timedelta(freq_two)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:344: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset_one = to_offset(freq_one)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:387: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  factor = pd.to_timedelta(freq_one) / pd.to_timedelta(freq_two)



2024-09-18 18:36:59,424: Epoch 3 average validation loss: 0.01929 -- Median validation metrics: avg_loss: 0.01929, NSE: 0.73417, KGE: 0.62684, Alpha-NSE: 0.69081, Beta-NSE: -0.00901, MSE: 0.81483, RMSE: 0.90268, Pearson-r: 0.87684, Beta-KGE: 0.83123, FHV: -28.58017, FMS: 24.40819, FLV: 0.00000, Peak-Timing: 0.46154, Missed-Peaks: 0.40000, Peak-MAPE: 67.07916
# Epoch 4: 100%|██████████| 13/13 [00:00<00:00, 25.49it/s, Loss: 0.0150]
2024-09-18 18:36:59,937: Epoch 4 average loss: avg_loss: 0.01095, avg_total_loss: 0.01095
# Epoch 5: 100%|██████████| 13/13 [00:00<00:00, 25.37it/s, Loss: 0.0073]
2024-09-18 18:37:00,456: Epoch 5 average loss: avg_loss: 0.01026, avg_total_loss: 0.01026
# Epoch 6: 100%|██████████| 13/13 [00:00<00:00, 25.08it/s, Loss: 0.0027]
2024-09-18 18:37:00,981: Epoch 6 average loss: avg_loss: 0.01083, avg_total_loss: 0.01083
# Validation: 100%|██████████| 1/1 [00:00<00:00,  3.45it/s]

/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:344: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset_one = to_offset(freq_one)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:387: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  factor = pd.to_timedelta(freq_one) / pd.to_timedelta(freq_two)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:344: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset_one = to_offset(freq_one)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:387: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  factor = pd.to_timedelta(freq_one) / pd.to_timedelta(freq_two)



2024-09-18 18:37:01,470: Epoch 6 average validation loss: 0.01925 -- Median validation metrics: avg_loss: 0.01925, NSE: 0.73472, KGE: 0.63774, Alpha-NSE: 0.69092, Beta-NSE: -0.00766, MSE: 0.81315, RMSE: 0.90175, Pearson-r: 0.87720, Beta-KGE: 0.85641, FHV: -28.40552, FMS: 26.26009, FLV: 0.00000, Peak-Timing: 0.46154, Missed-Peaks: 0.37778, Peak-MAPE: 66.81546
# Epoch 7: 100%|██████████| 13/13 [00:00<00:00, 24.63it/s, Loss: 0.0118]
2024-09-18 18:37:02,001: Epoch 7 average loss: avg_loss: 0.01024, avg_total_loss: 0.01024
# Epoch 8: 100%|██████████| 13/13 [00:00<00:00, 24.59it/s, Loss: 0.0262]
2024-09-18 18:37:02,537: Epoch 8 average loss: avg_loss: 0.01081, avg_total_loss: 0.01081
# Epoch 9: 100%|██████████| 13/13 [00:00<00:00, 25.23it/s, Loss: 0.0145]
2024-09-18 18:37:03,059: Epoch 9 average loss: avg_loss: 0.00998, avg_total_loss: 0.00998
# Validation: 100%|██████████| 1/1 [00:00<00:00,  3.46it/s]

/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:344: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset_one = to_offset(freq_one)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:387: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  factor = pd.to_timedelta(freq_one) / pd.to_timedelta(freq_two)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:344: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset_one = to_offset(freq_one)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:387: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  factor = pd.to_timedelta(freq_one) / pd.to_timedelta(freq_two)



2024-09-18 18:37:03,549: Epoch 9 average validation loss: 0.01922 -- Median validation metrics: avg_loss: 0.01922, NSE: 0.73504, KGE: 0.64270, Alpha-NSE: 0.69082, Beta-NSE: -0.00697, MSE: 0.81216, RMSE: 0.90120, Pearson-r: 0.87746, Beta-KGE: 0.86940, FHV: -28.33667, FMS: 27.24518, FLV: 0.00000, Peak-Timing: 0.46154, Missed-Peaks: 0.37778, Peak-MAPE: 66.66908
# Epoch 10: 100%|██████████| 13/13 [00:00<00:00, 24.55it/s, Loss: 0.0084]
2024-09-18 18:37:04,082: Epoch 10 average loss: avg_loss: 0.01015, avg_total_loss: 0.01015


In [3]:
config_file_path = Path(os.path.abspath('')) / 'runs' / f'basin_{basin}' / 'config.yml'
finetuned_model = TrainedModel(config_file_path_or_experiment_name=config_file_path)
evaluate_models([model, finetuned_model], basins=[basin], include_benchmark=False, period='train')
evaluate_models([model, finetuned_model], basins=[basin], include_benchmark=False, period='validation')
evaluate_models([model, finetuned_model], basins=[basin], include_benchmark=False, period='test')

/home/admin/Fine-Flood-Forecasts/neuralhydrology/evaluation/tester.py:133: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.model.load_state_dict(torch.load(weight_file, m

# Evaluation:   0%|          | 0/531 [00:00<?, ?it/s]

/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:344: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset_one = to_offset(freq_one)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:387: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  factor = pd.to_timedelta(freq_one) / pd.to_timedelta(freq_two)


# Evaluation:   0%|          | 1/531 [00:01<16:09,  1.83s/it]

/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:344: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset_one = to_offset(freq_one)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:387: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  factor = pd.to_timedelta(freq_one) / pd.to_timedelta(freq_two)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:344: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset_one = to_offset(freq_one)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:387: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  factor = pd.to_timedelta(freq_one) / pd.to_timedelta(freq_two)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:344: FutureWarning: 'H' is depreca

...

/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:344: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset_one = to_offset(freq_one)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:387: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  factor = pd.to_timedelta(freq_one) / pd.to_timedelta(freq_two)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:344: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset_one = to_offset(freq_one)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:387: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  factor = pd.to_timedelta(freq_one) / pd.to_timedelta(freq_two)


# Evaluation:  75%|███████▍  | 397/531 [04:19<01:14,  1.80it/s]

/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:344: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset_one = to_offset(freq_one)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:387: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  factor = pd.to_timedelta(freq_one) / pd.to_timedelta(freq_two)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:344: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset_one = to_offset(freq_one)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:387: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  factor = pd.to_timedelta(freq_one) / pd.to_timedelta(freq_two)


# Evaluation:  75%|███████▍  | 398/531 [04:19<01:14,  1.78it/s]

/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:344: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset_one = to_offset(freq_one)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:387: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  factor = pd.to_timedelta(freq_one) / pd.to_timedelta(freq_two)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:344: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  offset_one = to_offset(freq_one)
/home/admin/Fine-Flood-Forecasts/neuralhydrology/datautils/utils.py:387: FutureWarning: 'H' is deprecated and will be removed in a future version. Please use 'h' instead of 'H'.
  factor = pd.to_timedelta(freq_one) / pd.to_timedelta(freq_two)
